# HuggingFace Login

In [ ]:
# Run this cell to login to Hugging Face inside the notebook

from huggingface_hub import notebook_login
notebook_login()

# Corpus and training dataset

In [ ]:
from datasets import load_dataset, concatenate_datasets
from tqdm import tqdm
from transformers import AutoTokenizer
from collections import defaultdict

# Column names
source = "source"
text = "text"

# Source names
s_sciq = "allenai/sciq"
s_pubmed = "qiaojin/PubMedQA"
s_arc = "allenai/ai2_arc"
s_arc_easy = f"{s_arc} (ARC-Easy)"
s_arc_challenge = f"{s_arc} (ARC-Challenge)"
s_aquarat = "deepmind/aqua_rat"

MAX_TK = 512 # Max token length of one entire "text" item
MIN_ANS = 10 # Minimum string length after "H: " or after "A: " inside a "text" item

print("Loading datasets")
sciq = load_dataset(s_sciq, split="train") # ~ 11k
# Need to filter out sciq examples that don't contain support data
sciq = sciq.filter(lambda x: "support" in x)
pubmed_ss = load_dataset("csv", data_files="pubmedqa.csv")["train"] # 20k
arc_easy = load_dataset(s_arc, "ARC-Easy", split="train") # ~ 2.25k
arc_challenge = load_dataset(s_arc, "ARC-Challenge", split="train") # ~ 1.12k
aquarat_ss = load_dataset("csv", data_files="aquarat.csv")["train"] # 20k

# To check that the token count isn't over 512
print("Loading tokeniser")
tokeniser = AutoTokenizer.from_pretrained("mgatti/MNLP_M3_mcqa_model")

def answer_ind(a):
    ALPH = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
    if a in ALPH:
        return ALPH.index(a)
    return -1

print("Mapping datasets for corpus")
# sciq: question, distractor1, distractor2, distractor3, correct_answer, support
m_sciq = sciq.map(lambda x: {text : f"Example of related question Q and answer A: Q: {x["question"]} A: {x["support"]}\nThe answer is {x["correct_answer"]}", source : s_sciq}, remove_columns=sciq.column_names)
# pubmed: question, context
m_pubmed = pubmed_ss.map(lambda x: {text : f"Example of related question Q and hint H: Q: {x["question"]} H: {x["answer"]}", source : s_pubmed}, remove_columns=pubmed_ss.column_names)
# arc (easy and challenge): id, question, choices, answerKey
m_arc_easy = arc_easy.map(lambda x: {text : f"Example of related question Q and answer A: Q: {x["question"]} A: {x["choices"]["text"][answer_ind(x["answerKey"])]}", source : s_arc_easy}, remove_columns=arc_easy.column_names)
m_arc_challenge = arc_challenge.map(lambda x: {text : f"Example of related question Q and answer A: Q: {x["question"]} A: {x["choices"]["text"][answer_ind(x["answerKey"])]}", source : s_arc_challenge}, remove_columns=arc_challenge.column_names)
# aquarat: question, answer
m_aquarat = aquarat_ss.map(lambda x: {text : f"Example of related question Q and answer A: Q: {x["question"]} A: {x["answer"]}", source : s_aquarat}, remove_columns=aquarat_ss.column_names)

ds_list = [m_sciq, m_pubmed, m_arc_easy, m_arc_challenge, m_aquarat]
ds_lens = defaultdict(list)

for ds in ds_list:
    ds_source = ds[0][source]
    ds_len = len(ds)
    ds_lens[ds_source] = ds_len
    print(f"Original length of {ds_source}: {ds_len}")

print("Concatenating datasets for corpus")
concat = concatenate_datasets(ds_list)
total_len = len(concat)
print(f"Length of {len(ds_list)} combined datasets before filtering by token length (max tokens = {MAX_TK}) and by minimum answer length ({MIN_ANS}): {total_len}")

# We want to also filter out examples that have extremely short answers (answers like "b. ")
concat_final = concat.filter(lambda x: len(tokeniser.tokenize(x[text])) <= MAX_TK 
                             and len(x[text].split("H: ")[-1]) >= MIN_ANS if "H: " in x[text] else len(x[text].split("A: ")[-1]) >= MIN_ANS)
final_len = len(concat_final)
print(f"Length of combined datasets after filtering: {final_len}")



In [ ]:
# Reuse jupyter variables saved by kernel from previous cell - this will bascially be a copy pasta with a few minor tweaks (oops that's not clean coding, shame)
print("Mapping datasets for training set")

qst = "question"
ctx = "context"
# sciq: question, distractor1, distractor2, distractor3, correct_answer, support
tr_sciq = sciq.map(lambda x: {qst : x["question"], ctx : x["support"], source : s_sciq}, remove_columns=sciq.column_names)
# pubmed: question, context
tr_pubmed = pubmed_ss.map(lambda x: {qst : x["question"], ctx : x["answer"], source : s_pubmed}, remove_columns=pubmed_ss.column_names)
# arc (easy and challenge): id, question, choices, answerKey
tr_arc_easy = arc_easy.map(lambda x: {qst : x["question"], ctx : (x["choices"]["text"][answer_ind(x["answerKey"])]), source : s_arc_easy}, remove_columns=arc_easy.column_names)
tr_arc_challenge = arc_challenge.map(lambda x: {qst : x["question"], ctx : (x["choices"]["text"][answer_ind(x["answerKey"])]), source : s_arc_challenge}, remove_columns=arc_challenge.column_names)
# aquarat: question, answer
tr_aquarat = aquarat_ss.map(lambda x: {qst : x["question"], ctx : x["answer"], source : s_aquarat}, remove_columns=aquarat_ss.column_names)

trds_list = [tr_sciq, tr_pubmed, tr_arc_easy, tr_arc_challenge, tr_aquarat]
trds_lens = defaultdict(list)

for trds in trds_list:
    trds_source = trds[0][source]
    trds_len = len(trds)
    trds_lens[ds_source] = trds_len
    print(f"Original length of {trds_source}: {trds_len}")

print("Concatenating datasets for training DS")
tr_concat = concatenate_datasets(trds_list)
total_trlen = len(tr_concat)
# No filtering based on token length for training DS
print(f"Length of {len(trds_list)} combined datasets: {total_trlen}")



In [ ]:
# Eval DS 3 (MMLU-Pro test set but formatted for lighteval)
# id | question | choices | answer
# id -> 'TIGER-Lab/MMLU-Pro' | question -> x['question'] | choices -> x['options'] | answer -> x['answer']

from datasets import load_dataset

print("Loading MMLU-PRO")
mmlu_pro = load_dataset('TIGER-Lab/MMLU-Pro', split='test')
print(f"Loaded {len(mmlu_pro)} test examples, mapping...")
mmlu_pro_fmt = mmlu_pro.map(lambda x: {'id' : 'TIGER-Lab/MMLU-Pro', 'question' : x['question'], 'choices' : x['options'], 'answer' : x['answer']}, remove_columns=mmlu_pro.column_names)
print("Mapping done, uploading to hub")
# mmlu_pro_fmt.push_to_hub("danthepol/rag-eval-fmt-mmlupro", split="test", private=False)